# City vs PIP: Crime Coverage Check

**Question:** If we filter King County sales by `city=='SEATTLE'` (2021–2025), would we have all crime records covered?

**Covered** = crime is within 1 km of at least one property sale.

We compare city-filter vs PIP-filter: how many crimes fall within 1 km of at least one sale under each approach.

In [32]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union

KINGCO_DATE_FORMAT = '%Y-%m-%d'
SPD_DATE_FORMAT = '%m/%d/%Y %I:%M:%S %p'
START_YEAR = 2021
RADIUS_KM = 1.0
radius_deg = RADIUS_KM / 111  # approx at Seattle latitude

## Load sales (city Seattle)

In [33]:
sales_seattle = pd.read_csv('kingco_sales.csv')
sales_seattle['sale_date'] = pd.to_datetime(sales_seattle['sale_date'], format=KINGCO_DATE_FORMAT, errors='coerce')
sales_seattle['sale_year'] = sales_seattle['sale_date'].dt.year
sales_seattle = sales_seattle[(sales_seattle['sale_year'] >= START_YEAR) & (sales_seattle['city'] == 'SEATTLE')].copy()
sales_seattle = sales_seattle.dropna(subset=['latitude', 'longitude'])
print(f'Sales (city==SEATTLE, {START_YEAR}+): {len(sales_seattle):,}')

Sales (city==SEATTLE, 2021+): 31,545


## Load crimes

In [34]:
crimes_df = pd.read_csv('SPD_Crime_Data__2008-Present.csv', low_memory=False)
crimes_df.head()

,Report Number,Report DateTime,Offense ID,Offense Date,NIBRS Group AB,NIBRS Crime Against Category,Offense Sub Category,Shooting Type Group,Block Address,Latitude,Longitude,Beat,Precinct,Sector,Neighborhood,Reporting Area,Offense Category,NIBRS Offense Code Description,NIBRS_offense_code
0,2018-903268,04/14/2018 10:55:00 PM,7626378669,04/14/2018 09:40:00 PM,A,PROPERTY,LARCENY-THEFT,-,14TH AVE NW / NW 52ND ST,47.66643106,-122.373590892794,B2,North,B,BALLARD SOUTH,2449,PROPERTY CRIME,Theft From Motor Vehicle,23F
1,2009-116925,04/07/2009 12:33:00 PM,7640542762,04/04/2009 09:00:00 PM,A,PROPERTY,LARCENY-THEFT,-,6XX BLOCK OF STEWART ST,47.61395514,-122.336584688501,M2,West,M,-,12783,PROPERTY CRIME,Theft From Motor Vehicle,23F
2,2013-068517,02/28/2013 10:12:00 AM,7633645932,02/27/2013 07:00:00 PM,A,PROPERTY,LARCENY-THEFT,-,36XX BLOCK OF 13TH AVE W,47.6530852,-122.373596946735,Q2,West,Q,-,1156,PROPERTY CRIME,Theft of Motor Vehicle Parts or Accessories,23G
3,2009-333992,09/21/2009 08:31:00 AM,7684397399,08/14/2009 09:00:00 PM,A,PROPERTY,"PROPERTY OFFENSES (INCLUDES STOLEN, DESTRUCTION)",-,42XX BLOCK OF BROOKLYN AVE NE,47.659069,-122.314344,U2,North,U,-,745,ALL OTHER,Destruction/Damage/Vandalism of Property,290
4,2023-921602,12/17/2023 10:22:27 AM,53438690244,12/16/2023 04:20:00 PM,A,PROPERTY,LARCENY-THEFT,-,3XX BLOCK OF N 138TH ST,47.72961851,-122.354322987378,N1,North,N,BITTERLAKE,5815,PROPERTY CRIME,Theft From Motor Vehicle,23F


In [35]:
chunks = []
for chunk in pd.read_csv('SPD_Crime_Data__2008-Present.csv', chunksize=100000, low_memory=False):
    chunk['Report DateTime'] = pd.to_datetime(chunk['Report DateTime'], format=SPD_DATE_FORMAT, errors='coerce')
    chunk['year'] = chunk['Report DateTime'].dt.year
    chunk = chunk[chunk['year'] >= START_YEAR]
    chunks.append(chunk)
crimes_df = pd.concat(chunks, ignore_index=True)
print(f'Crimes ({START_YEAR}+): {len(crimes_df):,}')

Crimes (2021+): 362,827


## Apply END_DATE cutoff

In [37]:
max_crime = crimes_df['Report DateTime'].max()
max_sales = sales_seattle['sale_date'].max()
END_DATE = (min(max_crime, max_sales)- pd.Timedelta(days=1)).normalize()

crimes_df = crimes_df[crimes_df['Report DateTime'] <= END_DATE].copy()
sales_seattle = sales_seattle[sales_seattle['sale_date'] <= END_DATE].copy()

print(f'END_DATE: {END_DATE.date()}')
print(f'Crimes (date <= END_DATE): {len(crimes_df):,}')
print(f'Sales (date <= END_DATE): {len(sales_seattle):,}')

END_DATE: 2025-05-23
Crimes (date <= END_DATE): 362,611
Sales (date <= END_DATE): 28,236


## Join: add crime_count_1km to sales

In [31]:
crimes_df['Latitude'] = pd.to_numeric(crimes_df['Latitude'], errors='coerce')
crimes_df['Longitude'] = pd.to_numeric(crimes_df['Longitude'], errors='coerce')
crimes_valid = crimes_df[
    (crimes_df['Latitude'] > 40) & (crimes_df['Latitude'] < 50) &
    (crimes_df['Longitude'] < -100) & (crimes_df['Longitude'] > -125)
].copy()
crime_coords = crimes_valid[['Latitude', 'Longitude']].values
crime_dates = crimes_valid['Report DateTime'].values

crime_tree = cKDTree(crime_coords)

BATCH_SIZE = 50000
crime_counts = []
last_crime_dates = []
for i in range(0, len(sales_seattle), BATCH_SIZE):
    batch = sales_seattle.iloc[i:i+BATCH_SIZE]
    coords = batch[['latitude', 'longitude']].values
    sale_dates = batch['sale_date'].values
    all_indices = crime_tree.query_ball_point(coords, r=radius_deg)
    for j, inds in enumerate(all_indices):
        valid = [idx for idx in inds if crime_dates[idx] < sale_dates[j]]
        crime_counts.append(len(valid))
        last_crime_dates.append(max((crime_dates[idx] for idx in valid), default=pd.NaT))
    print(f'Processed {min(i+BATCH_SIZE, len(sales_seattle)):,} / {len(sales_seattle):,} properties...')

sales_seattle['crime_count_1km'] = crime_counts
sales_seattle['last_crime_date'] = last_crime_dates
df_joined = sales_seattle.copy()
sales_city = df_joined.copy()  # already Seattle (city filter)
print(f'\ndf_joined: {len(df_joined):,} rows (Seattle, city filter)')

Processed 28,241 / 28,241 properties...

df_joined: 28,241 rows (Seattle, city filter)


## Filter to Seattle (PIP)

In [ ]:
boundary_gdf = gpd.read_file('seattle_city_limits.geojson')
all_lines = []
for geom in boundary_gdf.geometry:
    if geom.geom_type == 'MultiLineString':
        for line in geom.geoms:
            all_lines.append(line)
    elif geom.geom_type == 'LineString':
        all_lines.append(geom)
lines_union = unary_union(all_lines)
seattle_boundary = lines_union.convex_hull

df_joined['geometry'] = df_joined.apply(lambda r: Point(r['longitude'], r['latitude']), axis=1)
gdf = gpd.GeoDataFrame(df_joined, geometry='geometry', crs='EPSG:4326')
in_seattle = gdf.geometry.within(seattle_boundary)
df_joined = df_joined[in_seattle].drop(columns=['geometry']).reset_index(drop=True)
sales_pip = df_joined.copy()
print(f'df_joined (Seattle, PIP): {len(df_joined):,} rows')

## Crime coverage: within 1 km of ≥1 sale

In [6]:
def count_crimes_covered(sales_df, crimes_df):
    """How many crimes are within 1 km of at least one sale?"""
    crimes_df = crimes_df.copy()
    crimes_df['Latitude'] = pd.to_numeric(crimes_df['Latitude'], errors='coerce')
    crimes_df['Longitude'] = pd.to_numeric(crimes_df['Longitude'], errors='coerce')
    crimes_valid = crimes_df[
        (crimes_df['Latitude'] > 40) & (crimes_df['Latitude'] < 50) &
        (crimes_df['Longitude'] < -100) & (crimes_df['Longitude'] > -125)
    ].copy()
    crime_coords = crimes_valid[['Latitude', 'Longitude']].values

    sale_coords = sales_df[['latitude', 'longitude']].values
    sale_tree = cKDTree(sale_coords)
    dists, _ = sale_tree.query(crime_coords, k=1, distance_upper_bound=radius_deg * 2)
    covered = dists <= radius_deg
    return covered.sum(), len(crimes_valid)

n_covered_city, n_crimes_valid = count_crimes_covered(sales_city, crimes_df)
n_covered_pip, _ = count_crimes_covered(sales_pip, crimes_df)

pct_city = 100 * n_covered_city / n_crimes_valid if n_crimes_valid > 0 else 0
pct_pip = 100 * n_covered_pip / n_crimes_valid if n_crimes_valid > 0 else 0

print(f'Crimes with valid Seattle-area coords: {n_crimes_valid:,}')
print()
print('Crime coverage (crime within 1 km of ≥1 sale):')
print(f'  City filter: {n_covered_city:,} / {n_crimes_valid:,} ({pct_city:.2f}%)')
print(f'  PIP filter:  {n_covered_pip:,} / {n_crimes_valid:,} ({pct_pip:.2f}%)')
print()
print(f'City filter misses {n_crimes_valid - n_covered_city:,} crimes ({100 - pct_city:.2f}%).')
print('Neither approach covers 100% (some crimes are >1 km from any sale).')

Crimes with valid Seattle-area coords: 308,380

Crime coverage (crime within 1 km of ≥1 sale):
  City filter: 271,761 / 308,380 (88.13%)
  PIP filter:  252,567 / 308,380 (81.90%)

City filter misses 36,619 crimes (11.87%).
Neither approach covers 100% (some crimes are >1 km from any sale).
